# mutascape demo

Synthetic combinatorial fitness landscapes on **CreiLOV**, a fluorescent LOV-domain protein with 118 mutable positions and real single-site DMS data.

## Load real data

In [ ]:
import numpy as np
import pandas as pd
from mutascape import ContactMap, Landscape, Mutant, RandomEpistasis, SignConsistentEpistasis

# Load pre-extracted CreiLOV contact map (from PDB 4I6G, chain A)
data = np.load("../data/contact_maps/creilov_distances.npz")
contacts = ContactMap.from_distance_matrix(
    data["distances"], data["positions"].tolist(), threshold=8.0
)

# WT sequence
from Bio import SeqIO
wt_seq = str(next(SeqIO.parse("../data/dms/creilov_wt.fasta", "fasta")).seq)

print(f"CreiLOV: {len(wt_seq)} residues, {contacts.n_contacts} structural contacts")
print(f"DMS data: {len(pd.read_csv('../data/dms/creilov.csv'))} single-site variants")

## Build landscape

In [ ]:
landscape = Landscape.from_dms(
    dms_path="../data/dms/creilov.csv",
    contact_map=contacts,
    epistasis=RandomEpistasis(sigma=0.3, seed=42),
    alpha=1.0,
    seed=42,
    wt_sequence=wt_seq,
)
landscape

## Querying the oracle

In [ ]:
# Single mutant: just the DMS fitness effect
landscape["A29H"]

In [ ]:
# Combine mutations with + operator
m = Mutant("A29H") + Mutant("R60D") + Mutant("F84W")
print(f"{m}")
print(f"  additive prediction: {landscape.additive_fitness(m):+.4f}")
print(f"  actual fitness:      {landscape[m]:+.4f}")

In [ ]:
# Full mutant sequence
landscape.mutant_sequence(m)

## Streaming with `generate()`

In [ ]:
# Sample 3-mutants on demand
for v in landscape.generate(n_mutations=3, n=5, seed=99):
    print(f"{v.mutations!s:30s}  fitness={v.fitness:+.4f}")

In [ ]:
# Infinite generator: search for a high-fitness variant
best = None
for i, v in enumerate(landscape.generate(n_mutations=4)):
    if best is None or v.fitness > best.fitness:
        best = v
    if best.fitness > 18.0 or i > 5000:
        break
print(f"Best after {i+1} samples: {best.mutations}  fitness={best.fitness:+.4f}")

## Bulk sampling

In [ ]:
df = landscape.to_dataframe(n_variants=5000, n_mutations=(1, 5), seed=42)
print(f"{len(df)} unique variants across depths {df['n_mutations'].min()}-{df['n_mutations'].max()}")
df.head(10)

## Landscape characterisation + plots

In [ ]:
landscape.stats(n_samples=2000, n_mutations=3)

In [ ]:
landscape.plot_contacts();

In [ ]:
landscape.plot_epistasis(depths=[1, 3, 5]);

In [ ]:
landscape.plot_fitness(depths=[1, 2, 3, 5]);

## Alpha sweep

In [ ]:
fig, sweep_data = landscape.plot_alpha_sweep()
pd.DataFrame(sweep_data).T

## Comparing epistasis models

In [ ]:
for name, model in [("Random", RandomEpistasis(sigma=0.3, seed=42)), ("SignConsistent", SignConsistentEpistasis(sigma=0.3, seed=42))]:
    ls = Landscape.from_dms("../data/dms/creilov.csv", contacts, model, alpha=1.0, seed=42, wt_sequence=wt_seq)
    s = ls.stats(n_samples=2000, n_mutations=3)
    print(f"{name:16s}  corr={s['additive_correlation']:.3f}  sign_epi={s['fraction_sign_epistasis']:.3f}")